# Week 2 · Day 1 — Modern CNN Architectures
**ResNet, EfficientNet & Pretrained Backbones**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

| | |
|---|---|
| **Runtime** | GPU (T4 — free tier) |
| **Estimated time** | ~30–45 min |
| **Topics** | ResNet residual connections · EfficientNet compound scaling · Pretrained weights API · Filter & feature-map viz · Inference benchmark |

---
### ⚡ Before you start
Go to **Runtime → Change runtime type → T4 GPU** then **Run All**.

## 0 · Setup & GPU Check

In [ ]:
# Verify GPU
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nUsing device: {DEVICE}')

In [ ]:
# Install/upgrade packages (Colab already has torch & torchvision)
!pip install -q wandb
print('✅ Packages ready')

In [ ]:
# Mount Google Drive (optional — to save results persistently)
SAVE_TO_DRIVE = False   # ← set True to persist results

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/ComputerVision/week2_classification/results'
else:
    RESULTS_DIR = '/content/results'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}')

## 1 · Imports

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.models import (
    ResNet18_Weights, ResNet50_Weights, ResNet101_Weights,
    EfficientNet_B0_Weights, EfficientNet_B3_Weights,
)
import torchvision.transforms as T
from torchvision import datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110
import numpy as np
import time
import urllib.request
from pathlib import Path

print('✅ Imports done')

## 2 · Helper: count parameters & load backbone

In [ ]:
BACKBONES = {
    'resnet18':       (models.resnet18,       ResNet18_Weights.DEFAULT),
    'resnet50':       (models.resnet50,       ResNet50_Weights.DEFAULT),
    'resnet101':      (models.resnet101,      ResNet101_Weights.DEFAULT),
    'efficientnet_b0':(models.efficientnet_b0, EfficientNet_B0_Weights.DEFAULT),
    'efficientnet_b3':(models.efficientnet_b3, EfficientNet_B3_Weights.DEFAULT),
}

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def load_backbone(name, pretrained=True):
    builder, weights = BACKBONES[name]
    model = builder(weights=weights if pretrained else None)
    return model.eval().to(DEVICE)

print('✅ Helpers defined')

## 3 · Architecture Summary
We trace each named child module, count its parameters, and capture output shapes via a forward hook.

In [ ]:
def print_architecture_summary(model, name, input_size=(1, 3, 224, 224)):
    print(f'\n{"="*55}')
    print(f'  Architecture: {name.upper()}')
    print(f'{"="*55}')
    print(f'  {"Module":<28} {"Parameters":>12}')
    print('  ' + '-'*42)
    total = 0
    for child_name, child in model.named_children():
        params = sum(p.numel() for p in child.parameters())
        total += params
        print(f'  {child_name:<28} {params:>12,}')
    print('  ' + '-'*42)
    print(f'  {"TOTAL":<28} {total:>12,}')

    # Shape trace
    shapes = {}
    def make_hook(n):
        def hook(module, inp, out):
            if isinstance(out, torch.Tensor):
                shapes[n] = tuple(out.shape)
        return hook
    handles = []
    for n, m in model.named_modules():
        handles.append(m.register_forward_hook(make_hook(n)))
    dummy = torch.zeros(input_size, device=DEVICE)
    with torch.no_grad():
        model(dummy)
    for h in handles:
        h.remove()

    print(f'\n  {"Layer":<28} {"Output Shape"}')
    print('  ' + '-'*50)
    for n in [cn for cn, _ in model.named_children()]:
        if n in shapes:
            print(f'  {n:<28} {str(shapes[n])}')


# --- Run it ---
for backbone_name in ['resnet18', 'efficientnet_b0']:
    m = load_backbone(backbone_name)
    print_architecture_summary(m, backbone_name)
    del m

## 4 · Residual Connections — Why They Work

In [ ]:
print("""
RESIDUAL CONNECTIONS — WHY THEY WORK
=====================================
Key problem (pre-ResNet):
  Very deep networks degraded in *training* accuracy because gradients
  vanished before reaching early layers.

ResNet solution — the shortcut (skip) connection:
  output = F(x) + x
  where F(x) is a stack of conv→BN→ReLU layers.

Benefits:
  1. Gradient highway: gradients flow back directly through '+',
     bypassing the conv stack → no vanishing gradient for the identity path.
  2. Residual learning: layers only need to learn the *difference* from
     identity — easy to optimise near zero.
  3. Ensemble effect: each block can be 'skipped' → network behaves like
     an ensemble of many shallower sub-networks.

Two block variants:
  • BasicBlock  (ResNet-18/34)  — two 3×3 convs, fast
  • Bottleneck  (ResNet-50/101) — 1×1→3×3→1×1, more efficient at depth
""")

# Inspect actual PyTorch implementation
from torchvision.models.resnet import BasicBlock, Bottleneck
print(f'BasicBlock param count (in=64, out=64) : {count_parameters(BasicBlock(64, 64)):,}')
print(f'Bottleneck  param count (in=256, out=64): {count_parameters(Bottleneck(256, 64)):,}')

In [ ]:
# Visualise the skip connection schematically
fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_title('Residual Block — Schematic', fontsize=13, fontweight='bold')

# Boxes
for x, label, color in [(1, 'x (input)', '#4C72B0'),
                         (3.5, 'Conv→BN→ReLU\nConv→BN', '#55A868'),
                         (7, 'output = F(x)+x\n(+ReLU)', '#C44E52')]:
    ax.add_patch(plt.FancyBboxPatch((x-0.9, 1.8), 1.8, 1.2,
                                     boxstyle='round,pad=0.1',
                                     facecolor=color, alpha=0.3, edgecolor=color, lw=2))
    ax.text(x, 2.4, label, ha='center', va='center', fontsize=9)

# Arrows (main path)
for x1, x2 in [(1.9, 2.6), (4.4, 6.1)]:
    ax.annotate('', xy=(x2, 2.4), xytext=(x1, 2.4),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))

# Skip connection
ax.annotate('', xy=(6.1, 3.5), xytext=(1.9, 3.5),
            arrowprops=dict(arrowstyle='->', color='#C44E52', lw=2, linestyle='dashed'))
ax.annotate('', xy=(7, 3.0), xytext=(7, 3.5),
            arrowprops=dict(arrowstyle='->', color='#C44E52', lw=1.5))
ax.text(4, 3.8, 'Skip connection (identity / 1×1 conv)', ha='center', color='#C44E52', fontsize=9)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/day1_residual_block.png', dpi=120)
plt.show()
print(f'Saved → {RESULTS_DIR}/day1_residual_block.png')

## 5 · EfficientNet — Compound Scaling

In [ ]:
print("""
EFFICIENTNET — COMPOUND SCALING
================================
EfficientNet scales three dimensions together with one coefficient φ:

    depth      d = α^φ
    width      w = β^φ
    resolution r = γ^φ

where α·β²·γ² ≈ 2  (double FLOPs per φ step).

B0 (baseline) → B7 (largest):
  B0: 5.3 M params,  224 px
  B3: 12 M params,   300 px   ← sweet spot for most tasks
  B7: 66 M params,   600 px
""")

# Show param counts for B0 and B3
for name in ['efficientnet_b0', 'efficientnet_b3']:
    m = load_backbone(name)
    print(f'  {name:<20}: {count_parameters(m)/1e6:.1f} M parameters')
    del m

## 6 · Inference Benchmark

In [ ]:
def benchmark(backbone_names, n_runs=30, batch_size=16):
    print(f'\nBenchmark  (batch={batch_size}, runs={n_runs}, device={DEVICE})')
    print(f'  {"Model":<20} {"Params (M)":>12} {"ms/img":>10} {"img/s":>10}')
    print('  ' + '-'*56)
    results = {}
    for name in backbone_names:
        model = load_backbone(name)
        img_size = 300 if 'b3' in name else 224
        dummy = torch.randn(batch_size, 3, img_size, img_size, device=DEVICE)

        with torch.no_grad():    # warm-up
            for _ in range(3): model(dummy)
        if DEVICE.type == 'cuda': torch.cuda.synchronize()

        t0 = time.perf_counter()
        with torch.no_grad():
            for _ in range(n_runs): model(dummy)
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

        total_imgs = n_runs * batch_size
        ms_per_img = elapsed / total_imgs * 1000
        throughput  = total_imgs / elapsed
        params_m    = count_parameters(model) / 1e6
        results[name] = dict(params_m=params_m, ms_per_img=ms_per_img, throughput=throughput)
        print(f'  {name:<20} {params_m:>12.1f} {ms_per_img:>10.2f} {throughput:>10.0f}')
        del model
    return results

bench = benchmark(['resnet18', 'resnet50', 'efficientnet_b0', 'efficientnet_b3'])

In [ ]:
# Bar chart — throughput comparison
names = list(bench.keys())
throughputs = [bench[n]['throughput'] for n in names]
params = [bench[n]['params_m'] for n in names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#4C72B0', '#55A868', '#C44E52', '#DD8452']

ax1.bar(names, throughputs, color=colors)
ax1.set_title('Throughput (images/sec) — Higher is Better')
ax1.set_ylabel('img/s')
ax1.tick_params(axis='x', rotation=20)

ax2.bar(names, params, color=colors)
ax2.set_title('Parameter Count (M) — Smaller = Lighter')
ax2.set_ylabel('Params (M)')
ax2.tick_params(axis='x', rotation=20)

plt.suptitle(f'Backbone Comparison on {DEVICE}', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/day1_benchmark.png', dpi=120)
plt.show()
print(f'Saved → {RESULTS_DIR}/day1_benchmark.png')

## 7 · Load a Sample Image (CIFAR-10)

In [ ]:
transform = T.Compose([
    T.Resize(224),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

dataset = datasets.CIFAR10(root='/content/data', train=False, download=True, transform=transform)
img_tensor, label = dataset[0]
img_tensor = img_tensor.unsqueeze(0).to(DEVICE)

# Show the raw (unnormalised) image
raw_img = datasets.CIFAR10(root='/content/data', train=False, download=False)[0][0]
plt.imshow(raw_img)
plt.title(f'Sample image: {dataset.classes[label]}')
plt.axis('off')
plt.show()
print(f'Class: {dataset.classes[label]}  |  Tensor shape: {img_tensor.shape}')

## 8 · Visualise First-Layer Conv Filters

In [ ]:
def visualise_filters(model, model_name):
    if hasattr(model, 'conv1'):
        filters = model.conv1.weight.data.cpu()
    else:
        try:
            filters = model.features[0][0].weight.data.cpu()
        except Exception:
            print('Could not locate first conv layer'); return

    f_min, f_max = filters.min(), filters.max()
    filters = (filters - f_min) / (f_max - f_min + 1e-8)

    n = min(32, filters.shape[0])
    cols, rows = 8, (n + 7) // 8
    fig, axes = plt.subplots(rows, cols, figsize=(cols*1.3, rows*1.3))
    fig.suptitle(f'{model_name} — First-Layer Filters', fontsize=11)
    for i, ax in enumerate(axes.flat):
        if i < n:
            ax.imshow(np.clip(filters[i].permute(1,2,0).numpy(), 0, 1))
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/day1_{model_name}_filters.png', dpi=120)
    plt.show()

for name in ['resnet18', 'efficientnet_b0']:
    m = load_backbone(name)
    visualise_filters(m, name)
    del m

## 9 · Visualise Feature Maps (first block output)

In [ ]:
def visualise_feature_maps(model, model_name, img_t, n_maps=16):
    captured = {}
    target = model.layer1 if hasattr(model, 'layer1') else model.features[1]
    def hook(m, inp, out): captured['feat'] = out.detach().cpu()
    handle = target.register_forward_hook(hook)
    with torch.no_grad(): model(img_t)
    handle.remove()

    feat = captured['feat'][0]
    n = min(n_maps, feat.shape[0])
    cols, rows = 8, (n + 7) // 8
    fig, axes = plt.subplots(rows, cols, figsize=(cols*1.3, rows*1.3))
    fig.suptitle(f'{model_name} — Feature Maps (block 1)', fontsize=11)
    for i, ax in enumerate(axes.flat):
        if i < n:
            ax.imshow(feat[i].numpy(), cmap='viridis')
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/day1_{model_name}_feature_maps.png', dpi=120)
    plt.show()

for name in ['resnet18', 'efficientnet_b0']:
    m = load_backbone(name)
    visualise_feature_maps(m, name, img_tensor)
    del m

## 10 · ImageNet Top-5 Inference Demo

In [ ]:
import urllib.request

labels_url = 'https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt'
urllib.request.urlretrieve(labels_url, '/content/imagenet_classes.txt')
with open('/content/imagenet_classes.txt') as f:
    imagenet_labels = [l.strip() for l in f]

def top5_inference(model, img_t, model_name):
    with torch.no_grad():
        logits = model(img_t)
    probs = logits.softmax(-1)[0]
    top_probs, top_ids = probs.topk(5)
    print(f'\n  Top-5 predictions [{model_name}]:')
    for p, idx in zip(top_probs, top_ids):
        print(f'    {imagenet_labels[idx.item()]:<40} {p.item()*100:5.1f}%')

for name in ['resnet18', 'resnet50', 'efficientnet_b0']:
    m = load_backbone(name)
    top5_inference(m, img_tensor, name)
    del m

## 11 · Architecture Comparison Table

In [ ]:
print('\nArchitecture Comparison Summary')
print(f'{"="*68}')
print(f'  {"Model":<20} {"Params (M)":>12} {"ms/img":>10} {"img/s":>10}  Notes')
print('  ' + '-'*66)
notes = {
    'resnet18':       'Lightweight, great baseline',
    'resnet50':       'Standard workhorse',
    'efficientnet_b0':'Tiny but strong',
    'efficientnet_b3':'Best param-efficiency (Day 2 pick)',
}
for name, r in bench.items():
    note = notes.get(name, '')
    print(f'  {name:<20} {r["params_m"]:>12.1f} {r["ms_per_img"]:>10.2f} {r["throughput"]:>10.0f}  {note}')

print('\n✅ Day 1 complete!')
print('   Next → Week2_Day2_Transfer_Learning.ipynb')